# Notebook 3 – Prepare the Reference Proteome

fDOG-Assembly validates candidate orthologs with a **backward BLAST search**:
the predicted protein from the assembly is BLASTed against a reference proteome,
and accepted only if the best hit matches the gene ID stored in the core group.

In Notebook 1 we used the OMA **canonical ID** as the gene identifier.
For human GAPDH this is `G3P_HUMAN` — the UniProt entry name.
The reference proteome must use the same ID as the sequence identifier.

The UniProt FASTA header looks like:
```
>sp|P04406|G3P_HUMAN Glyceraldehyde-3-phosphate dehydrogenase ...
```
We keep only the **third field** (`G3P_HUMAN`) as the sequence ID.
Because it contains no pipe character, `fdog.addTaxon` leaves it unchanged,
so the BLAST database entry will be exactly `G3P_HUMAN` — matching the core group.

---
### Prerequisites
| Requirement | How to obtain |
|---|---|
| fDOG installed | `pip install fdog` |
| biopython | `pip install biopython` |
| BLAST+ | system package manager |

## 1 – Configuration

In [15]:
import gzip
import shutil
import subprocess
from pathlib import Path
from Bio import SeqIO

import requests

# ── Reference species ─────────────────────────────────────────────────────────
REF_SPECIES_CODE = "HUMAN"    # must match Notebook 1
NCBI_TAXON_ID    = "9606"
VERSION          = "OMA2024"  # must match the version used in Notebook 1

# ── Output ────────────────────────────────────────────────────────────────────
DATA_DIR     = Path("data")
FDOG_REF_DIR = DATA_DIR / "fdog_reference"
FDOG_REF_DIR.mkdir(parents=True, exist_ok=True)

print(f"Reference species : {REF_SPECIES_CODE} (taxon {NCBI_TAXON_ID})")
print(f"Version tag       : {VERSION}")
print(f"Output directory  : {FDOG_REF_DIR}")

Reference species : HUMAN (taxon 9606)
Version tag       : OMA2024
Output directory  : data/fdog_reference


## 2 – Download the Human Reference Proteome from UniProt

The UniProt reference proteome for *Homo sapiens* (UP000005640) contains
~20,000 reviewed protein sequences.

In [16]:
UNIPROT_URL = (
    "https://ftp.uniprot.org/pub/databases/uniprot/current_release/"
    "knowledgebase/reference_proteomes/Eukaryota/UP000005640/"
    "UP000005640_9606.fasta.gz"
)
gz_path = DATA_DIR / "UP000005640_9606.fasta.gz"

if gz_path.exists():
    print(f"Already downloaded: {gz_path}")
else:
    print("Downloading UniProt human reference proteome ...")
    with requests.get(UNIPROT_URL, stream=True, timeout=300) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        downloaded = 0
        with open(gz_path, "wb") as fh:
            for chunk in r.iter_content(chunk_size=1_048_576):
                fh.write(chunk)
                downloaded += len(chunk)
                if total:
                    print(f"  {downloaded/1e6:.1f} / {total/1e6:.1f} MB",
                          end="\r", flush=True)
    print(f"\nSaved: {gz_path}  ({gz_path.stat().st_size/1e6:.1f} MB)")

Already downloaded: data/UP000005640_9606.fasta.gz


## 3 – Convert Headers to UniProt Entry Names

Original header:
```
>sp|P04406|G3P_HUMAN Glyceraldehyde-3-phosphate dehydrogenase ...
```
We extract the third `|`-field (`G3P_HUMAN`) as the sequence ID.
This matches the `canonicalid` that OMA returned in Notebook 1.

In [17]:
clean_fasta = DATA_DIR / "HUMAN_uniprot.fa"

converted = 0
with gzip.open(gz_path, "rt") as fin, open(clean_fasta, "w") as fout:
    for record in SeqIO.parse(fin, "fasta"):
        # record.id is e.g. 'sp|P04406|G3P_HUMAN'
        parts = record.id.split("|")
        entry_name = parts[2] if len(parts) >= 3 else record.id
        fout.write(f">{entry_name}\n{record.seq}\n")
        converted += 1

print(f"Converted {converted} sequences → {clean_fasta}")
print()
# Preview a few headers to confirm
count = 0
with open(clean_fasta) as fh:
    for line in fh:
        if line.startswith(">"):
            print(f"  {line.rstrip()}")
            count += 1
        if count >= 4:
            break

Converted 20652 sequences → data/HUMAN_uniprot.fa

  >LV469_HUMAN
  >CS2IP_HUMAN
  >A0A1B0GVY7_HUMAN
  >A0A590UJ96_HUMAN


## 4 – Register with `fdog.addTaxon`

- `-c` : build a BLAST database in `coreTaxa_dir` (required for backward validation)
- `-a` : skip FAS annotation (not needed for fDOG-Assembly)
- `-o` : write to our local `data/fdog_reference/` instead of the global fDOG data folder

In [18]:
cmd = [
    "fdog.addTaxon",
    "-f", str(clean_fasta),
    "-i", NCBI_TAXON_ID,
    "-o", str(FDOG_REF_DIR),
    "-n", REF_SPECIES_CODE,
    "-v", VERSION,
    "-c",
    "-a",
]

print("Running:", " ".join(cmd))
print()
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
    raise RuntimeError(f"fdog.addTaxon failed (exit {result.returncode})")

Running: fdog.addTaxon -f data/HUMAN_uniprot.fa -i 9606 -o data/fdog_reference -n HUMAN -v OMA2024 -c -a



Building a new DB, current time: 07/25/2026 22:27:47
New DB name:   /home/hannah/Dev/fdog-assembly/fDOG-Assembly/examples/data/fdog_reference/coreTaxa_dir/HUMAN@9606@OMA2024/HUMAN@9606@OMA2024
New DB title:  /home/hannah/Dev/fdog-assembly/fDOG-Assembly/examples/data/fdog_reference/searchTaxa_dir/HUMAN@9606@OMA2024/HUMAN@9606@OMA2024.fa
Sequence type: Protein
Keep MBits: T
Maximum file size: 3000000000B
Adding sequences from FASTA; added 20652 sequences in 1.41094 seconds.


Species name	HUMAN@9606@OMA2024
Parsing FASTA file...

Creating Blast DB...

==> Output for HUMAN@9606@OMA2024 can be found in /home/hannah/Dev/fdog-assembly/fDOG-Assembly/examples/data/fdog_reference/searchTaxa_dir, /home/hannah/Dev/fdog-assembly/fDOG-Assembly/examples/data/fdog_reference/coreTaxa_dir



## 5 – Verify the Output

In [19]:
species_name = f"{REF_SPECIES_CODE}@{NCBI_TAXON_ID}@{VERSION}"

expected = {
    "Proteome FASTA" : FDOG_REF_DIR / "searchTaxa_dir" / species_name / f"{species_name}.fa",
    "BLAST DB (.phr)": FDOG_REF_DIR / "coreTaxa_dir"   / species_name / f"{species_name}.phr",
}

all_ok = True
for label, path in expected.items():
    ok = path.exists()
    print(f"  [{'OK' if ok else 'MISSING'}] {label:20}  {path}")
    all_ok &= ok

print()
if all_ok:
    print("Reference proteome ready!")
    print()
    print("Use in fdog.assembly (Notebook 4):")
    print(f"  --dataPath {FDOG_REF_DIR}")
    print(f"  --refSpec  {species_name}")
else:
    print("Some files are missing – check the fdog.addTaxon output above.")

  [OK] Proteome FASTA        data/fdog_reference/searchTaxa_dir/HUMAN@9606@OMA2024/HUMAN@9606@OMA2024.fa
  [OK] BLAST DB (.phr)       data/fdog_reference/coreTaxa_dir/HUMAN@9606@OMA2024/HUMAN@9606@OMA2024.phr

Reference proteome ready!

Use in fdog.assembly (Notebook 4):
  --dataPath data/fdog_reference
  --refSpec  HUMAN@9606@OMA2024


## Summary

| Step | Result |
|---|---|
| Download | `data/UP000005640_9606.fasta.gz` |
| Convert headers | `data/HUMAN_uniprot.fa` — IDs like `G3P_HUMAN` |
| `fdog.addTaxon -c -a` | `data/fdog_reference/coreTaxa_dir/HUMAN@9606@OMA2024/` |

**Next step:** `04_run_fdog_assembly.ipynb`